# Multi-Rat Sliding-Window Decoding Curve Validation (All 5 Rats)

**Purpose:** notebook 008 found that a later window (around +1475ms for InSeq/OutSeq, +1025ms for Odor
Identity) meaningfully outperforms the fixed 500ms-at-Poke-In window used everywhere before it, on Mitt.
Before adopting that as a new default, or building a full multi-rat pooled training pipeline, this
notebook checks whether the same pattern holds across all 5 rats, or whether it was specific to Mitt.

**Approach:** the exact same sliding-window scan from notebook 008 (250ms window, 25ms steps, -500ms to
+1500ms relative to Poke-In, spectral features with the updated bands), run separately on each of the 5
rats. Rats are kept SEPARATE here, not pooled into one combined dataset, this notebook is about checking
generalization of the window-timing pattern, not yet about combining data for more training examples,
that's a following step once this is reviewed.

**Runtime note:** this trains and evaluates 81 window positions x 2 tasks x 5 rats = a meaningfully
larger computation than notebook 008 alone. Expect this to take several minutes, not seconds, let it run.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from src.preprocessing import build_labels, get_sampling_rate


## 1. Identify All Sessions

In [ ]:
raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(session_dirs)} sessions:")
for p in session_dirs:
    print(" ", p.name)


## 2. Shared Settings (Same as Notebook 008)

In [ ]:
WINDOW_LENGTH_MS = 250
STEP_MS = 25
SPAN_START_MS = -500
SPAN_END_MS = 1500

offsets_ms = np.arange(SPAN_START_MS, SPAN_END_MS + 1, STEP_MS)

BANDS = {
    'delta': (1, 4),
    'theta': (4, 12),
    'beta': (12, 30),
    'low_gamma': (30, 80),
    'high_gamma': (80, 150),
}

print(f"{len(offsets_ms)} window positions per rat")


## 3. Helper Functions (Same as Notebook 008)

In [ ]:
def extract_window_at_offset(lfp_data, timebin, poke_idx, offset_ms, window_ms, target_samples):
    poke_time = timebin[poke_idx]
    start_time = poke_time + offset_ms / 1000
    end_time = start_time + window_ms / 1000

    start_idx = np.searchsorted(timebin, start_time)
    end_idx = np.searchsorted(timebin, end_time)
    if end_idx - start_idx < 2:
        return None

    raw_window = lfp_data[:, start_idx:end_idx]
    n_channels, n_raw = raw_window.shape
    old_x = np.linspace(0, 1, n_raw)
    new_x = np.linspace(0, 1, target_samples)
    resampled = np.zeros((n_channels, target_samples))
    for ch in range(n_channels):
        resampled[ch] = np.interp(new_x, old_x, raw_window[ch])
    return resampled


def band_power_features(windows, fs, bands):
    n_trials, n_channels, n_samples = windows.shape
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    band_masks = {name: (freqs >= lo) & (freqs < hi) for name, (lo, hi) in bands.items()}
    n_features = n_channels * len(bands)
    X = np.zeros((n_trials, n_features))
    col = 0
    for ch in range(n_channels):
        fft_vals = np.fft.rfft(windows[:, ch, :], axis=1)
        power = np.abs(fft_vals) ** 2
        for band_name, mask in band_masks.items():
            X[:, col] = power[:, mask].sum(axis=1)
            col += 1
    return X


def run_sliding_window_for_session(session_dir, offsets_ms, window_length_ms, bands):
    """Runs the full sliding-window scan for one rat's session, returns both decoding curves."""
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    lfp = np.load(session_dir / f'{session_name}_lfp.npz', allow_pickle=True)
    lfp_data = lfp['data']

    timebin = bvr_data[bvr_keys.index('TimeBin')]
    avg_fs = get_sampling_rate(bvr_data, bvr_keys)
    labels = build_labels(bvr_data, bvr_keys)
    target_samples = int(round(window_length_ms / 1000 * avg_fs))

    min_time_needed_before = abs(offsets_ms[0]) / 1000
    max_time_needed_after = (offsets_ms[-1] + window_length_ms) / 1000

    valid_trial_idx = []
    for t in labels['trial_idx']:
        poke_time = timebin[t]
        if poke_time - min_time_needed_before < timebin[0]:
            continue
        if poke_time + max_time_needed_after > timebin[-1]:
            continue
        valid_trial_idx.append(t)
    valid_trial_idx = np.array(valid_trial_idx)
    valid_mask = np.isin(labels['trial_idx'], valid_trial_idx)
    inseq_outseq = labels['inseq_outseq'][valid_mask]
    odor_id = labels['odor_id'][valid_mask]

    inseq_curve, odor_curve = [], []
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for offset in offsets_ms:
        windows = []
        for t in valid_trial_idx:
            w = extract_window_at_offset(lfp_data, timebin, t, offset, window_length_ms, target_samples)
            windows.append(w)
        windows = np.stack(windows, axis=0)
        X = np.log1p(band_power_features(windows, avg_fs, bands))

        pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
        inseq_scores = cross_val_score(pipe, X, inseq_outseq, cv=skf, scoring='balanced_accuracy')
        inseq_curve.append(inseq_scores.mean())

        inseq_mask_local = (inseq_outseq == 1)
        X_odor = X[inseq_mask_local]
        y_odor = odor_id[inseq_mask_local]
        skf_odor = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        odor_scores = cross_val_score(pipe, X_odor, y_odor, cv=skf_odor, scoring='balanced_accuracy')
        odor_curve.append(odor_scores.mean())

    return {
        'session': session_name,
        'n_trials': len(valid_trial_idx),
        'inseq_curve': np.array(inseq_curve),
        'odor_curve': np.array(odor_curve),
    }


## 4. Run the Sliding Window Scan for Every Rat

This is the long-running cell in this notebook, it repeats notebook 008's full analysis 5 times, once
per rat. Progress is printed after each rat finishes.


In [ ]:
all_results = {}
for session_dir in session_dirs:
    print(f"Running {session_dir.name}...")
    result = run_sliding_window_for_session(session_dir, offsets_ms, WINDOW_LENGTH_MS, BANDS)
    all_results[session_dir.name] = result
    print(f"  done, {result['n_trials']} trials used, "
          f"InSeq/OutSeq best={result['inseq_curve'].max():.3f}, "
          f"Odor best={result['odor_curve'].max():.3f}")

print("\nAll rats complete.")


## 5. Visualize Results Across Rats

The overlapping line charts from the first version of this notebook were hard to read with 5 rats at
once. Replaced with three clearer views:

1. **Heatmaps** (rat x window offset, color = accuracy): makes it easy to scan for which rats/offsets
   are strong, without 5 tangled lines competing for attention.
2. **Peak accuracy bar chart**: directly compares each rat's single best score per task, side by side,
   this is the fastest way to see that Superchris leads on InSeq/OutSeq.
3. **Smoothed line curves** (optional, kept for anyone who wants the fine-grained shape): the raw curves
   are noisy point-to-point, a rolling average makes the underlying trend easier to see than the raw
   version from notebook 008/009 v1.


In [ ]:
session_names = list(all_results.keys())
n_rats = len(session_names)

inseq_matrix = np.array([all_results[s]['inseq_curve'] for s in session_names])  # (n_rats, n_offsets)
odor_matrix = np.array([all_results[s]['odor_curve'] for s in session_names])

# --- 1. Heatmaps ---
fig, axes = plt.subplots(2, 1, figsize=(13, 7))

im0 = axes[0].imshow(inseq_matrix, aspect='auto', cmap='viridis', vmin=0.4, vmax=0.9,
                      extent=[offsets_ms[0], offsets_ms[-1], n_rats - 0.5, -0.5])
axes[0].set_yticks(range(n_rats))
axes[0].set_yticklabels(session_names)
axes[0].axvline(0, color='white', linestyle=':', alpha=0.7)
axes[0].set_xlabel('Window start time relative to Poke-In (ms)')
axes[0].set_title('InSeq/OutSeq balanced accuracy (all rats)')
plt.colorbar(im0, ax=axes[0], label='balanced accuracy')

im1 = axes[1].imshow(odor_matrix, aspect='auto', cmap='viridis', vmin=0.2, vmax=0.5,
                      extent=[offsets_ms[0], offsets_ms[-1], n_rats - 0.5, -0.5])
axes[1].set_yticks(range(n_rats))
axes[1].set_yticklabels(session_names)
axes[1].axvline(0, color='white', linestyle=':', alpha=0.7)
axes[1].set_xlabel('Window start time relative to Poke-In (ms)')
axes[1].set_title('Odor Identity balanced accuracy (all rats)')
plt.colorbar(im1, ax=axes[1], label='balanced accuracy')

plt.tight_layout()
plt.show()


In [ ]:
# --- 2. Peak accuracy bar chart, directly answers "which rats do best" ---
inseq_peaks = [all_results[s]['inseq_curve'].max() for s in session_names]
odor_peaks = [all_results[s]['odor_curve'].max() for s in session_names]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_rats)
width = 0.35
bars1 = ax.bar(x - width/2, inseq_peaks, width, label='InSeq/OutSeq (chance=0.5)', color='#4C72B0')
bars2 = ax.bar(x + width/2, odor_peaks, width, label='Odor Identity (chance=0.2)', color='#DD8452')
ax.axhline(0.85, color='green', linestyle='--', alpha=0.5, label='target (~0.85)')
ax.set_xticks(x)
ax.set_xticklabels(session_names, rotation=20, ha='right')
ax.set_ylabel('Peak balanced accuracy (best of 81 windows tested)')
ax.set_title('Peak decoding accuracy per rat')
ax.legend()
for bar in list(bars1) + list(bars2):
    height = bar.get_height()
    ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# --- 3. Smoothed line curves (rolling average), easier to read trend than raw noisy points ---
def smooth(curve, window=5):
    kernel = np.ones(window) / window
    return np.convolve(curve, kernel, mode='same')

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_rats))

for session_name, color in zip(session_names, colors):
    axes[0].plot(offsets_ms, smooth(all_results[session_name]['inseq_curve']), label=session_name, color=color, linewidth=2)
axes[0].axhline(0.5, color='black', linestyle='--', alpha=0.4)
axes[0].axvline(0, color='black', linestyle=':', alpha=0.6)
axes[0].set_title('InSeq/OutSeq, smoothed (5-point rolling average)')
axes[0].set_xlabel('Offset from Poke-In (ms)')
axes[0].set_ylabel('Balanced accuracy')
axes[0].legend(fontsize=8)

for session_name, color in zip(session_names, colors):
    axes[1].plot(offsets_ms, smooth(all_results[session_name]['odor_curve']), label=session_name, color=color, linewidth=2)
axes[1].axhline(0.2, color='black', linestyle='--', alpha=0.4)
axes[1].axvline(0, color='black', linestyle=':', alpha=0.6)
axes[1].set_title('Odor Identity, smoothed (5-point rolling average)')
axes[1].set_xlabel('Offset from Poke-In (ms)')
axes[1].set_ylabel('Balanced accuracy')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 6. Why Does Performance Vary So Much By Rat? A Quick Signal-Quality Check

Superchris scores notably higher than the others on InSeq/OutSeq (0.898 vs. 0.695-0.810). Before treating
that as "Superchris's brain signal is just better," two things need checking: (1) whether it's a real,
robust difference or an artifact of picking the single best score out of 81 tested windows (a
multiple-comparisons effect that hits harder for rats with fewer minority-class trials), and (2) trial
counts and class balance per rat, which affect how noisy each rat's cross-validation estimate is.


In [ ]:
print(f"{'Rat':20s} {'Total trials':14s} {'OutSeq trials':14s} {'OutSeq %':10s}")
print("-" * 60)
for session_name, result in all_results.items():
    n_total = result['n_trials']
    # recompute OutSeq count for this rat directly, using the same labels already loaded during the scan
    session_dir = [p for p in session_dirs if p.name == session_name][0]
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    labels_check = build_labels(bvr_data, bvr_keys)
    n_outseq = int((labels_check['inseq_outseq'] == 0).sum())
    pct_outseq = 100 * n_outseq / len(labels_check['inseq_outseq'])
    print(f"{session_name:20s} {n_total:<14d} {n_outseq:<14d} {pct_outseq:<10.1f}")

print("\nWith only 5-fold CV, each fold's test set holds roughly 1/5 of the OutSeq trials.")
print("A rat with ~20-30 OutSeq trials total has only ~4-6 per test fold, small enough that a couple")
print("of trials landing favorably can noticeably inflate a single fold's (and thus the best-of-81-search's) score.")


## 7. Per-Rat Summary Table

Best offset and best balanced accuracy per rat, per task, plus the hypothesis check (does odor peak
before InSeq/OutSeq) evaluated separately for every rat, not just Mitt.


In [ ]:
print(f"{'Rat':20s} {'InSeq best':12s} {'@ offset':10s} {'Odor best':12s} {'@ offset':10s} {'Odor peaks first?':18s}")
print("-" * 85)

summary_rows = []
for session_name, result in all_results.items():
    inseq_best_idx = result['inseq_curve'].argmax()
    odor_best_idx = result['odor_curve'].argmax()
    inseq_best = result['inseq_curve'][inseq_best_idx]
    odor_best = result['odor_curve'][odor_best_idx]
    inseq_offset = offsets_ms[inseq_best_idx]
    odor_offset = offsets_ms[odor_best_idx]
    odor_first = odor_offset < inseq_offset

    print(f"{session_name:20s} {inseq_best:<12.3f} {inseq_offset:<10d} {odor_best:<12.3f} {odor_offset:<10d} {str(odor_first):18s}")

    summary_rows.append({
        'session': session_name,
        'inseq_best_balanced_accuracy': round(float(inseq_best), 4),
        'inseq_best_offset_ms': int(inseq_offset),
        'odor_best_balanced_accuracy': round(float(odor_best), 4),
        'odor_best_offset_ms': int(odor_offset),
        'odor_peaks_before_inseq': bool(odor_first),
    })


## 8. Text-Only Results Export


In [ ]:
import json as _json
import os

results_summary = {
    "purpose": "Multi-rat validation of notebook 008's sliding-window finding",
    "window_length_ms": WINDOW_LENGTH_MS,
    "step_ms": STEP_MS,
    "span_ms": [int(SPAN_START_MS), int(SPAN_END_MS)],
    "bands": {name: list(rng) for name, rng in BANDS.items()},
    "per_rat_summary": summary_rows,
    "n_rats_where_odor_peaks_first": sum(row['odor_peaks_before_inseq'] for row in summary_rows),
    "n_rats_total": len(summary_rows),
}

print(_json.dumps(results_summary, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook009_results.json', 'w') as f:
    _json.dump(results_summary, f, indent=2)
print("\nSaved to outputs/logs/notebook009_results.json")


## 9. Written Summary Report

**Objective**

Check whether the sliding-window finding from notebook 008 (later windows outperform the fixed 500ms
window; Odor Identity peaks before InSeq/OutSeq) generalizes across all 5 rats, or was specific to Mitt.

**Method**

The exact sliding-window scan from notebook 008 (250ms window, 25ms steps, -500 to +1500ms relative to
Poke-In, spectral features with updated bands) repeated independently on each of the 5 rats. Rats were
NOT pooled into one dataset, each rat's curves were computed entirely separately.

**Results**

Peak InSeq/OutSeq balanced accuracy by rat: Mitt 69.5%, Barat 81.0%, Stella 80.7%, Superchris 89.8%,
Buchanan 71.3%. Peak Odor Identity balanced accuracy: Mitt 42.8%, Barat 46.1%, Stella 49.0%,
Superchris 46.1%, Buchanan 46.7%. All 5 rats (5 of 5) show the hypothesized "Odor Identity peaks before
InSeq/OutSeq" pattern, though the specific best offsets vary considerably, three rats (Barat, Stella,
Buchanan) show their single best Odor Identity score at a NEGATIVE offset (before Poke-In), which is
biologically implausible on its face, since the odor hasn't been presented yet at that point.

**Interpretation**

Two findings, one encouraging and one that needs caution before being taken at face value.

Encouraging: InSeq/OutSeq decoding is meaningfully above chance for every rat, and for 3 of 5 rats
(Barat, Stella, Superchris) the peak score is close to or exceeds the ~85% target, using a later window
than the fixed 500ms-at-Poke-In default used in every notebook before 008. This strongly suggests window
timing was a real limiting factor, not just for Mitt.

Needs caution: Mitt, the rat every prior notebook was built and tuned around, is actually the WEAKEST
performer for InSeq/OutSeq of all 5 rats. This is worth stating plainly rather than glossing over,
earlier notebooks' conclusions were drawn from what turns out to be a below-average case.

Also needs caution: the pre-Poke-In "best" Odor Identity offsets for 3 rats are very likely a
multiple-comparisons artifact, not a real finding, the odor hasn't been presented before Poke-In, so
any above-chance score there should be noise. This is a direct consequence of searching for the single
maximum across 81 tested window positions per rat: with OutSeq/minority-class trial counts as small as
21-44 per rat, and only ~4-6 per cross-validation fold, some noisy peaks are expected to appear by
chance, including implausible ones. None of the specific "best offset" numbers reported here should be
treated as final without validation (see Next Steps).

**Next Steps**

1. Validate the top few offsets per rat (not just the single argmax) using multiple random CV seeds, to
   separate real signal from search noise, this is especially important given the implausible
   pre-Poke-In "best" results for Odor Identity in 3 rats.
2. Investigate why Mitt underperforms relative to the other 4 rats specifically, before assuming it's
   representative, possible causes include lower minority-class trial count relative to some other
   rats, electrode placement/signal quality differences, or session-specific behavioral variability.
3. Once validated, decide whether to adopt one shared window across all rats or rat-specific windows,
   then build the actual multi-rat pooled training pipeline with a session-based train/test split.
4. Run a permutation test on the validated results before presenting a final number for Sep 1-2.
